In [1]:
%pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.7 MB/s  0:00:011.5 MB/s eta 0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
from datasets import load_from_disk
ds = load_from_disk("../data/clean")

with open("../data/spm_train.txt", "w", encoding="utf-8") as f:
    for line in ds["train"]:
        f.write(line["en"] + "\n")
        f.write(line["ru"] + "\n")

In [7]:
import sentencepiece as spm
spm.SentencePieceTrainer.train(
    input="../data/spm_train.txt",
    model_prefix="bpe",
    vocab_size=16000,
    model_type="bpe",
    character_coverage=1.0,
    byte_fallback=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
)

I0000 00:00:1794855561.784826 3966412 sentencepiece_trainer.cc:149] Starts training with : 
trainer_spec {
  input: ../data/spm_train.txt
  input_format: 
  model_prefix: bpe
  model_type: BPE
  vocab_size: 16000
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  auto_character_coverage: 0
  vocabulary_output_piece_score: 1
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
}
normalizer_spec {
  name: nmt_nfkc
  add_dummy

In [9]:
sp = spm.SentencePieceProcessor(model_file="bpe.model")
ids = sp.encode("You should have accepted his advice.")
print(ids, sp.decode(ids))

[416, 778, 427, 9407, 549, 3381, 15768] You should have accepted his advice.


In [11]:
for line in ds["train"].select(range(20)):
    ids_en = sp.encode(line["en"])
    ids_ru = sp.encode(line["ru"])
    print(ids_en, sp.decode(ids_en))
    print(ids_ru, sp.decode(ids_ru))

[416, 778, 427, 9407, 549, 3381, 15768] You should have accepted his advice.
[1230, 5233, 8629, 735, 597, 8660, 15768] Тебе стоило последовать его совету.
[416, 1902, 533, 1397, 2791, 549, 3381, 15768] You may as well follow his advice.
[442, 1546, 332, 8629, 735, 597, 8660, 15768] Ты можешь и последовать его совету.
[2270, 362, 286, 427, 279, 366, 350, 279, 2791, 549, 3381, 15768] All that you have to do is to follow his advice.
[2647, 15799, 318, 459, 1147, 771, 15799, 360, 8629, 735, 597, 8660, 15768] Всё, что ты должен сделать, это последовать его совету.
[2270, 362, 286, 427, 279, 366, 350, 279, 2791, 549, 3381, 15768] All that you have to do is to follow his advice.
[2647, 15799, 318, 402, 1507, 771, 15799, 360, 8629, 735, 597, 8660, 15768] Всё, что вы должны сделать, это последовать его совету.
[416, 1276, 2298, 4650, 279, 514, 15768] You must pay attention to him.
[442, 1147, 1880, 5838, 320, 1167, 6281, 15768] Ты должен обращать на него внимание.
[416, 1276, 1293, 2282, 810, 5

In [13]:
import numpy as np

sample = ds["train"].shuffle(seed=0).select(range(20000))
en_ids = sp.encode(list(sample["en"]))
ru_ids = sp.encode(list(sample["ru"]))

In [14]:
en_len = np.array([len(x) for x in en_ids])
ru_len = np.array([len(x) for x in ru_ids])
print(en_len.mean(), ru_len.mean(), ru_len.mean() / en_len.mean())

8.49955 7.87905 0.9269961350895048


In [15]:
print(np.percentile(en_len, [50, 95, 99, 100]))
print(np.percentile(ru_len, [ 50, 95, 99, 100]))

[ 8. 14. 21. 76.]
[ 7. 14. 21. 88.]


In [16]:
total = sum(len(x) for x in en_ids + ru_ids)
unk = sum(i == sp.unk_id() for x in en_ids + ru_ids for i in x)
print(unk / total)

0.0


In [19]:
def tokenize(batch):
    return {
        "src": sp.encode(batch["en"]),
        "tgt": sp.encode(batch["ru"], add_bos=True, add_eos=True),
    }

ds = ds.map(tokenize, batched=True)

Map:   0%|          | 0/508012 [00:00<?, ? examples/s]

Map:   0%|          | 0/10370 [00:00<?, ? examples/s]

Map:   0%|          | 0/36380 [00:00<?, ? examples/s]

In [23]:
def filt(ex):
    if len(ex["src"]) > 48 or len(ex["tgt"]) > 50:
        return False
    return True

train = ds["train"].filter(filt)
print(len(train), len(ds["train"]), (len(ds["train"]) - len(train)) / len(train))

507684 508012 0.0006460711781344301


In [24]:
ds["train"] = train
ds.save_to_disk("../data/tokenized")

Saving the dataset (0/1 shards):   0%|          | 0/507684 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10370 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/36380 [00:00<?, ? examples/s]